# AgriSmart Shield – Phần 2: Huấn luyện trên Google Colab

Notebook chạy Bước 2–3 và phần kiểm thử của Bước 6: kiểm tra dữ liệu → huấn luyện YOLOv8n-seg → đánh giá → lượng tử hóa INT8 (ONNX, cho máy chủ trang web) → so sánh FP32/INT8 → kiểm thử độ bền → tạo bộ ảnh mẫu Demo.

**Chuẩn bị trước (làm một lần):**
1. Trên Google Drive tạo thư mục `MyDrive/AgriSmart`.
2. Tải lên thư mục đó:
   - `agrismart-shield.zip`: mã nguồn (nút "Tải toàn bộ mã nguồn" trong file HTML).
   - `agrismart_dataset.zip`: nén **thư mục** `datasets/agrismart` đã tạo ở Phần 1 (sau `s02`, `s03`, `s04`). Trong file zip phải có thư mục `agrismart/` chứa `images/`, `labels/`, `dataset.yaml`.
3. Menu **Runtime → Change runtime type → T4 GPU**.

Kết quả huấn luyện được lưu thẳng vào Drive (`MyDrive/AgriSmart/runs`), nên không mất khi Colab ngắt kết nối. Nếu bị ngắt giữa chừng, chạy ô **Huấn luyện tiếp**.

In [ ]:
# [Ô 1]
# Kiểm tra đã có GPU
!nvidia-smi

In [ ]:
# [Ô 2]
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/AgriSmart'
REPO_ZIP = f'{DRIVE}/agrismart-shield.zip'
DATA_ZIP = f'{DRIVE}/agrismart_dataset.zip'
RUNS = f'{DRIVE}/runs'
DATA_YAML = '/content/datasets/agrismart/dataset.yaml'
import os
for p in (REPO_ZIP, DATA_ZIP):
    print(('OK   ' if os.path.exists(p) else 'THIẾU'), p)

In [ ]:
# [Ô 3]
# Giải nén mã nguồn và cài thư viện (ultralytics dùng PyTorch có sẵn trên Colab)
!rm -rf /content/agrismart-shield && unzip -q "{REPO_ZIP}" -d /content
%cd /content/agrismart-shield
!pip install -q -r ai/requirements.txt
!python -c "import ultralytics, torch; print('ultralytics', ultralytics.__version__, '| torch', torch.__version__, '| CUDA', torch.cuda.is_available())"

In [ ]:
# [Ô 4]
# Giải nén bộ dữ liệu và sửa đường dẫn trong dataset.yaml
# (s02 ghi đường dẫn tuyệt đối của máy tính ở Phần 1, ví dụ C:\agrismart-shield\..., trên Colab phải đổi lại)
import pathlib, yaml
!mkdir -p /content/datasets && unzip -q -o "{DATA_ZIP}" -d /content/datasets
cfg_path = pathlib.Path(DATA_YAML)
if not cfg_path.exists():
    raise SystemExit('Không thấy ' + DATA_YAML + ' – kiểm tra file zip có thư mục agrismart/ ở gốc. Nội dung: ' + str(list(pathlib.Path('/content/datasets').glob('*'))))
cfg = yaml.safe_load(cfg_path.read_text(encoding='utf-8'))
cfg['path'] = str(cfg_path.parent)
cfg_path.write_text(yaml.safe_dump(cfg, allow_unicode=True, sort_keys=False), encoding='utf-8')
print(cfg)

In [ ]:
# [Ô 5]
# Bước 1 (kiểm tra lại): nhãn đúng định dạng, mỗi ảnh có lá, mọi lớp bệnh có quy tắc IPM
!python ai/scripts/s04_check_dataset.py --data-yaml "{DATA_YAML}"

## Bước 2 – Huấn luyện YOLOv8n-seg
Mất khoảng vài giờ trên T4, tùy số ảnh. Checkpoint được lưu mỗi 10 epoch vào Drive.

In [ ]:
# [Ô 6]
!python ai/scripts/s05_train.py --data "{DATA_YAML}" --epochs 100 --imgsz 640 --batch 16 --project "{RUNS}" --name agrismart_v8n --save-period 10

In [ ]:
# [Ô 7]
# Huấn luyện tiếp – CHỈ chạy ô này khi Colab bị ngắt giữa chừng (bỏ dấu # ở dòng dưới)
# !python ai/scripts/s05_train.py --data "{DATA_YAML}" --resume "{RUNS}/agrismart_v8n/weights/last.pt"

In [ ]:
# [Ô 8]
# Đánh giá mô hình gốc (FP32) trên tập test: Box/Mask mAP50-95, Mean IoU, sai số Severity
BEST = f'{RUNS}/agrismart_v8n/weights/best.pt'
!python ai/scripts/s06_evaluate.py --weights "{BEST}" --data "{DATA_YAML}" --split test --out reports/eval_fp32.json

## Bước 3 – Lượng tử hóa INT8 (ONNX cho máy chủ web) và so sánh

In [ ]:
# [Ô 9]
# Xuất ONNX INT8 (lượng tử hóa tĩnh bằng ONNX Runtime, hiệu chỉnh trên ảnh của dataset.yaml)
WEB = f'{DRIVE}/model_for_web'
!python ai/scripts/s07_export.py --weights "{BEST}" --data "{DATA_YAML}" --format onnx-int8 --copy-to "{WEB}"
!ls -la "{WEB}"

In [ ]:
# [Ô 10]
# Đánh giá bản INT8, so sánh với FP32 (tiêu chí mAP giảm ≤ 2 điểm %), đo tốc độ tham khảo trên CPU của Colab
!python ai/scripts/s06_evaluate.py --weights "{WEB}/model_int8.onnx" --data "{DATA_YAML}" --split test --out reports/eval_int8.json
!python ai/scripts/s08_benchmark.py compare --fp32 reports/eval_fp32.json --int8 reports/eval_int8.json --max-drop 2.0 --mode absolute --out reports/compare.json
!python ai/scripts/s08_benchmark.py latency --model "{WEB}/model_int8.onnx" --images /content/datasets/agrismart/images/test --runs 30 --out reports/latency_colab.json
!mkdir -p "{DRIVE}/reports" && cp reports/*.json "{DRIVE}/reports/"

## Bước 6 – Kiểm thử thay thực địa và bộ ảnh mẫu Demo
Không đi thực địa được: đo độ bền của mô hình với 10 điều kiện chụp mô phỏng (mờ, rung tay, thiếu/thừa sáng, chụp xa, nén ảnh, nghiêng, nhiễu) trên tập test, rồi tạo bộ ảnh mẫu có đáp án để trình diễn trên trang web. Chạy bằng CPU, vài phút.

In [ ]:
# [Ô 11]
!python ai/scripts/s09_robustness.py --model "{WEB}/model_int8.onnx" --data "{DATA_YAML}" --split test --out reports/robustness
!python ai/scripts/s10_demo_pack.py --data "{DATA_YAML}" --split test --n 24 --source-map plantseg_=plantseg --out demo/samples
!cd demo && zip -qr "{DRIVE}/demo_samples.zip" samples && cd ..
!cp reports/robustness.* "{DRIVE}/reports/" && ls -la "{DRIVE}"

## (Tùy chọn) Kiểm tra độc lập trên PlantDoc
Chỉ áp dụng khi mô hình có 3 lớp bệnh cà chua (`ai/configs/plantdoc_map.yaml`). Ảnh PlantDoc trùng với tập huấn luyện bị loại trước khi đánh giá.

In [ ]:
# [Ô 12]
!git clone -q https://github.com/pratikkayal/PlantDoc-Object-Detection-Dataset /content/plantdoc
!python ai/scripts/datasets_catalog.py dedup --a /content/plantdoc/TEST --b /content/datasets/agrismart/images/train --out reports/trung_plantdoc_train.csv
!python ai/scripts/s06b_eval_bbox.py --weights "{BEST}" --csv /content/plantdoc/test_labels.csv --images /content/plantdoc/TEST --map ai/configs/plantdoc_map.yaml --exclude-list reports/trung_plantdoc_train.csv --out reports/eval_plantdoc.json
!cp reports/eval_plantdoc.json "{DRIVE}/reports/"

## Việc tiếp theo (Phần 3 – trang web trên máy tính)
1. Tải thư mục `MyDrive/AgriSmart/model_for_web` về máy, chép `model_int8.onnx` và `model_meta.json` vào `web/models/` của dự án.
2. Tải `MyDrive/AgriSmart/demo_samples.zip`, giải nén vào thư mục `demo/` của dự án (được `demo/samples/`).
3. Tải thư mục `MyDrive/AgriSmart/reports` về `reports/` của dự án.
4. Chạy `python web/run.py`, mở trên điện thoại cùng mạng Wi-Fi theo địa chỉ in ra màn hình.

Số đo tốc độ ở ô 10 chạy trên CPU của Colab, chỉ để tham khảo. Tiêu chí < 100 ms đo lại trên máy tính chạy Demo bằng `s08_benchmark.py latency ... --out reports/latency.json`.